<a href="https://colab.research.google.com/github/vandana512/CiteWise/blob/main/CiteWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install PyPDF2 scikit-learn
!pip install pymupdf
!pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 60.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.1 MB/s eta 0:00:00


In [4]:
import re
import fitz
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files

# -------------------------------
# UPLOAD PDF
# -------------------------------
print("Upload PDF:")
uploaded = files.upload()
pdf_bytes = next(iter(uploaded.values()))

doc = fitz.open(stream=pdf_bytes, filetype="pdf")
text = "\n".join(page.get_text() for page in doc)
lines = [l.strip() for l in text.split("\n") if l.strip()]

# -------------------------------
# TITLE
# -------------------------------
title = "Unknown Title"
for line in lines[:40]:
    if any(x in line.lower() for x in ["doi","http","license","elsevier","issn"]):
        continue
    if 30 < len(line) < 200 and not re.search(r'\d{4}', line):
        title = line
        break

# -------------------------------
# AUTHORS (FINAL WORKING)
# -------------------------------
authors = []

# extract from first page only (important)
first_page = doc[0].get_text()

match = re.search(r"age of smart learning\s+(.*?)\s+a College", first_page, re.S)

if match:
    block = match.group(1)

    # remove commas + line breaks
    block = block.replace("\n", " ")

    authors = re.findall(r"[A-Z][a-z]+(?: [A-Z][a-z]+)+", block)

# clean
authors = list(dict.fromkeys(authors))

# -------------------------------
# YEAR
# -------------------------------
year_match = re.search(r'\b(20\d{2})\b', text)
year = year_match.group(1) if year_match else "Unknown"

# -------------------------------
# ABSTRACT (FINAL WORKING)
# -------------------------------
abstract = ""

match = re.search(r"ABSTRACT\s+(.*?)\s+1\.", text, re.S)

if match:
    abstract = match.group(1)

# clean garbage
abstract = re.sub(r"http\S+", "", abstract)
abstract = re.sub(r"creativecommons.*?license", "", abstract, flags=re.I)
abstract = re.sub(r"\s+", " ", abstract).strip()

# fallback
if len(abstract) < 50:
    abstract = " ".join(lines[15:25])

# -------------------------------
# KEYWORDS
# -------------------------------
clean_text = re.sub(r"[^a-z\s]", "", abstract.lower())

vectorizer = TfidfVectorizer(stop_words="english", max_features=15)
X = vectorizer.fit_transform([clean_text])
keywords = []
for w in vectorizer.get_feature_names_out():
    if len(w) < 5:
        continue
    if any(x in w for x in ["http", "license", "article"]):
        continue
    keywords.append(w)

keywords = keywords[:5]

# -------------------------------
#  CITATIONS [1], [2]
# -------------------------------
citations = set(re.findall(r'\[(\d+)\]', text))

citations = sorted(citations, key=int)

print(f"\n✓ Citations found: {len(citations)}")
# -------------------------------
#  REFERENCES (FORCED EXTRACTION)
# -------------------------------
references = {}

ref_section = re.search(r"References(.*)", text, re.I | re.S)

if ref_section:
    ref_lines = ref_section.group(1).split("\n")

    for line in ref_lines:
        match = re.match(r'\[(\d+)\]', line)
        if match:
            ref_id = match.group(1)
            references[ref_id] = line.strip()

print(f"✓ References extracted: {len(references)}")
# -------------------------------
# VALIDATION
# -------------------------------
valid = []
invalid = []

for c in citations:
    if c in references:
        valid.append(c)
    else:
        invalid.append(c)

# -------------------------------
# OUTPUT
# -------------------------------
print("\n📄 METADATA")
print("Title:", title)
print("Authors:", authors)
print("Year:", year)
print("Keywords:", keywords)
print("Abstract:", abstract[:400], "...")

print(f"\n✔ {len(valid)} VALID | ✖ {len(invalid)} INVALID")

print("\n✔ VALID CITATIONS (sample)")
for v in valid[:10]:
    print(f"[{v}] -> {references[v][:80]}...")

print("\n❌ INVALID CITATIONS")
print(invalid[:10])

Upload PDF:


Saving The_Effect_of_Procrastination_on_Physical_Exercise_among_College_StudentsThe_Ch.pdf to The_Effect_of_Procrastination_on_Physical_Exercise_among_College_StudentsThe_Ch.pdf

✓ Citations found: 27
✓ References extracted: 3

📄 METADATA
Title: The Effect of Procrastination on Physical
Authors: []
Year: 2024
Keywords: ['action', 'activity', 'behavior', 'college', 'commitment']
Abstract: Background: Exercise procrastination is prevalent among college students, causing decline in physical ﬁtness. It is imperative to investigate the mechanism affecting college students’ physical activity behaviors. This study was aimed at investigating the effect of procrastination on college students’ physical exercise behavior, and the chain mediation effects of exercise commitment and action cont ...

✔ 3 VALID | ✖ 24 INVALID

✔ VALID CITATIONS (sample)
[31] -> [31]. The study concluded that exercise commitment, as a...
[61] -> [61]....
[64] -> [64]...

❌ INVALID CITATIONS
['2', '3', '5', '6', '11', '